Dataset 2 Initial Inspection
Project objective
This project investigates whether room occupancy can be predicted from environmental sensor measurements using machine-learning methods.

Available data
The dataset contains:

Occupancy sensor records
Environmental sensor records
Environmental sensor and room location information
Inspection goals
Identify the exact raw files and formats
Examine columns and data types
Inspect missing and duplicate values
Examine occupancy categories
Parse timestamps
Extract environmental measurements from JSON
Determine how occupancy, environmental sensors and rooms can be joined

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   #importing data

raw_folder = Path("../data/raw") 
print("Raw folder exists:", raw_folder.exists())
print("Raw folder location:", raw_folder.resolve())  #Creating pathway for raw data folder

raw_files = sorted(
    file
    for file in raw_folder.iterdir()
    if file.is_file() and file.name != ".gitkeep"
)

for file in raw_files:
    file_size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {file_size_mb:.2f} MB")   #Listing all raw files and their sizes in MB

"""
csv_files = list(raw_folder.glob("*.csv"))

for file in csv_files:
    print("\n" + "=" * 80)
    print(f"FILE: {file.name}")

    sample = pd.read_csv(
        file,
        nrows=5,
        low_memory=False
    )

    print("Sample shape:", sample.shape)
    print("Columns:")
    print(sample.columns.tolist())

    display(sample)           #Creating sample of each csv file to inspect the data and its columns
"""
occupancy_path = raw_folder / "5occupancySensor_MayToDec2024_9MRows.csv"
environment_path = raw_folder / "5EnvSensor_MayToDec2024_180kRows.csv"
locations_path = raw_folder / "Sensor ID and Locations (1).xlsx"

print("Occupancy file exists:", occupancy_path.exists())
print("Environmental file exists:", environment_path.exists())
print("Location file exists:", locations_path.exists())  #Creating pathway for each of the three main files and checking if they exist

occupancy_sample = pd.read_csv(
    occupancy_path,
    nrows=100_000,
    low_memory=False
)

environment_data = pd.read_csv(
    environment_path,
    low_memory=False   
)
location_data = pd.read_excel(locations_path)   #loading data from the three main files into pandas dataframes


def clean_column_names(dataframe):
    """Return a copy with standardised column names."""
    cleaned = dataframe.copy()

    cleaned.columns = (
        cleaned.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    return cleaned

occupancy_sample = clean_column_names(occupancy_sample)
environment_data = clean_column_names(environment_data)
location_data = clean_column_names(location_data) #cleaning up columns

print("Occupancy columns:")
print(occupancy_sample.columns.tolist())

print("\nEnvironmental columns:")
print(environment_data.columns.tolist())

print("\nLocation columns:")
print(location_data.columns.tolist())

occupancy_counts = (
    occupancy_sample["occupancystatus"]
    .value_counts(dropna=False)   
)
display(occupancy_counts) #Displaying occupancy counts for each status in the sample data

occupancy_percentages = (
    occupancy_sample["occupancystatus"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)
display(occupancy_percentages) #Displaying occupancy percentages for each status in the sample data

headcount_counts = (
    occupancy_sample["headcount"]
    .value_counts(dropna=False)
    .sort_index()
)
display(headcount_counts) #displaying headcount counts for each value in the sample data
#Currently occupied responds to headcount 1 and everything else is 0. So we can use headcount as a proxy for occupancy status.

for value in environment_data["jsondata"].dropna().head(5):
    print(value)
    print("-" * 100) #Prints full environmental records

import json
example_json = environment_data["jsondata"].dropna().iloc[0]
parsed_example = json.loads(example_json)

print("Original value:")
print(example_json)

print("\nParsed type:")
print(type(parsed_example))

print("\nParsed content:")
print(parsed_example) #Parsing JSON Data from the environmental data to inspect its structure and content



In [ ]:
example_variables = pd.DataFrame([
    {
        "variable_id": item["variable"]["id"],
        "variable_name": item["variable"]["name"],
        "unit": item["variable"]["unit"],
        "value": item["values"][0]["value"],
        "unix_time": item["values"][0]["time"]
    }
    for item in parsed_example
])

display(example_variables)  #Example of readable table for environmental observation

example_unix_time = parsed_example[0]["values"][0]["time"]

example_measurement_time = pd.to_datetime(
    example_unix_time,
    unit="s",
    utc=True
)

print("Unix timestamp:", example_unix_time)
print("Converted timestamp:", example_measurement_time) #Converting Unix timestamp to human-readable format

example_createdate = pd.to_datetime(
    environment_data["createdate"].iloc[0],
    errors="coerce",
    utc=True
)

example_processdate = pd.to_datetime(
    environment_data["processdate"].iloc[0],
    errors="coerce",
    utc=True
)

print("Measurement time:", example_measurement_time)
print("Create date:", example_createdate)
print("Process date:", example_processdate)

Environmental JSON structure
Each jsondata record contains a list of environmental measurements. Each measurement contains a variable name, unit, numerical value and Unix timestamp.

The inspected record contains carbon dioxide, humidity, formaldehyde, pressure, temperature, volatile organic compounds and several particulate matter measurements. Battery voltage is also recorded but is primarily a sensor-health variable rather than an environmental predictor.

These measurements are potential explanatory variables for predicting current room occupancy.

In [ ]:
occupancy_rooms = (
    occupancy_sample[
        ["deviceid", "floorspaceid"]
    ]
    .drop_duplicates()
    .sort_values(
        ["floorspaceid", "deviceid"]
    )
)

print("Unique occupancy devices:", occupancy_rooms["deviceid"].nunique())
print("Unique floor spaces:", occupancy_rooms["floorspaceid"].nunique())
display(occupancy_rooms.head(30))

workbook = pd.ExcelFile(locations_path)

print("Workbook sheets:", workbook.sheet_names)

for sheet_name in workbook.sheet_names:
    sheet = pd.read_excel(
        locations_path,
        sheet_name=sheet_name
    )

    print("\n" + "=" * 80)
    print("Sheet:", sheet_name)
    print("Columns:", sheet.columns.tolist())
    display(sheet.head(10))        #Checking contents of occupancy identifiers and location workbook to see if there are any additional identifiers or location information that can be used to join the occupancy and environmental data   